In [1]:
import pandas as pd

In [2]:
start_url = 'https://www.london-fire.gov.uk/community/your-borough/'

In [3]:
import re
import time
from urllib.parse import urljoin, urlparse, parse_qs

import requests
from bs4 import BeautifulSoup

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
})

def fetch_soup(url):
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def get_area_urls(start_url):
    soup = fetch_soup(start_url)
    area_items = []
    seen = set()
    for a in soup.select("a[href]"):
        href = a.get("href")
        if not href:
            continue
        full = urljoin(start_url, href)
        if "/community/" not in full:
            continue
        parsed = urlparse(full)
        if not parsed.netloc.endswith("london-fire.gov.uk"):
            continue
        path = parsed.path.rstrip("/")
        if not (path.count("/") == 2 and path.startswith("/community/")):
            continue
        area_name = a.get_text(strip=True)
        if not area_name:
            continue
        key = (area_name, full)
        if key in seen:
            continue
        seen.add(key)
        area_items.append({"area": area_name, "url": full})
    return area_items

def is_location_link(text):
    if not text:
        return False
    t = text.strip().lower()
    return t in {"get directions", "get location", "get locations"}

def parse_fire_stations(area_url, area_name):
    soup = fetch_soup(area_url)
    stations = []
    # Find station cards via location links
    for link in soup.select("a[href]"):
        if not is_location_link(link.get_text(strip=True)):
            continue
        container = link.find_parent(["div", "li", "article", "section"]) or link.parent
        heading = None
        if container:
            heading = container.find(["h2", "h3", "h4"])
        if not heading:
            heading = link.find_previous(["h2", "h3", "h4"])
        name = heading.get_text(strip=True) if heading else None
        text_lines = [t.strip() for t in container.stripped_strings] if container else []
        text_lines = [
            t for t in text_lines
            if t.lower() not in {"get directions", "get location", "get locations", "contact us"}
        ]
        text_block = " ".join(text_lines)
        # UK postcode regex
        postcode = None
        m = re.search(r"\b([A-Z]{1,2}\d{1,2}[A-Z]?\s*\d[A-Z]{2})\b", text_block)
        if m:
            postcode = m.group(1).replace(" ", "")
            # normalize with space before last 3 chars
            postcode = postcode[:-3] + " " + postcode[-3:]
        stations.append({
            "name": name,
            "area": area_name,
            "postcode": postcode
        })
    return stations

area_urls = get_area_urls(start_url)
print(f"Found {len(area_urls)} area pages")
area_urls

Found 40 area pages


[{'area': 'Community Engagement',
  'url': 'https://www.london-fire.gov.uk/community/community-engagement/'},
 {'area': 'Young people',
  'url': 'https://www.london-fire.gov.uk/community/young-people/'},
 {'area': 'Public Notices',
  'url': 'https://www.london-fire.gov.uk/community/public-notices/'},
 {'area': 'Your borough',
  'url': 'https://www.london-fire.gov.uk/community/your-borough/'},
 {'area': 'Events & Open Days',
  'url': 'https://www.london-fire.gov.uk/community/events-open-days/'},
 {'area': 'Wellbeing Support',
  'url': 'https://www.london-fire.gov.uk/community/wellbeing-support/'},
 {'area': 'Barking and Dagenham',
  'url': 'https://www.london-fire.gov.uk/community/barking-and-dagenham/'},
 {'area': 'City',
  'url': 'https://www.london-fire.gov.uk/community/the-city-of-london/'},
 {'area': 'Hackney',
  'url': 'https://www.london-fire.gov.uk/community/hackney/'},
 {'area': 'Havering',
  'url': 'https://www.london-fire.gov.uk/community/havering/'},
 {'area': 'Islington',
 

In [4]:
all_stations = []
for idx, item in enumerate(area_urls, start=1):
    try:
        stations = parse_fire_stations(item["url"], item["area"])
        all_stations.extend(stations)
        print(f"[{idx}/{len(area_urls)}] {item['url']} -> {len(stations)} stations")
    except Exception as exc:
        print(f"Failed: {item['url']} -> {exc}")
    # Be polite to the server
    time.sleep(0.5)

stations_df = pd.DataFrame(all_stations)
stations_df = stations_df.drop_duplicates(subset=["name", "area", "postcode"]).reset_index(drop=True)

[1/40] https://www.london-fire.gov.uk/community/community-engagement/ -> 0 stations
[2/40] https://www.london-fire.gov.uk/community/young-people/ -> 0 stations
[3/40] https://www.london-fire.gov.uk/community/public-notices/ -> 0 stations
[4/40] https://www.london-fire.gov.uk/community/your-borough/ -> 0 stations
[5/40] https://www.london-fire.gov.uk/community/events-open-days/ -> 0 stations
[6/40] https://www.london-fire.gov.uk/community/wellbeing-support/ -> 0 stations
[7/40] https://www.london-fire.gov.uk/community/barking-and-dagenham/ -> 2 stations
[8/40] https://www.london-fire.gov.uk/community/the-city-of-london/ -> 1 stations
[9/40] https://www.london-fire.gov.uk/community/hackney/ -> 3 stations
[10/40] https://www.london-fire.gov.uk/community/havering/ -> 4 stations
[11/40] https://www.london-fire.gov.uk/community/islington/ -> 2 stations
[12/40] https://www.london-fire.gov.uk/community/newham/ -> 3 stations
[13/40] https://www.london-fire.gov.uk/community/redbridge/ -> 3 stati

In [5]:
stations_df

,name,area,postcode
0,Barking,Barking and Dagenham,IG11 0BB
1,Dagenham,Barking and Dagenham,RM10 7ES
2,Dowgate,City,EC4R 3UE
3,Shoreditch,Hackney,EC1V 9EY
4,Stoke Newington,Hackney,N16 0AR
...,...,...,...
98,Wandsworth,Wandsworth,SW18 1RL
99,Battersea,Wandsworth,SW11 2TL
100,Tooting,Wandsworth,SW17 7SQ
101,Paddington,Westminster,W2 6NL


The location information above is from the official links of 'get directions' in LFB website. However, the coordinates in the Google Map urls are actually the center of the map, which may not be the exact location of the fire station. Therefore, I will use the geopy library to obtain the geocodes of the stations based on their names.

In [6]:
# try to request the coordinates using station names and postcodes via Nominatim API
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
geolocator = Nominatim(user_agent="london_fire_stations_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
def geocode_station(row):
    query = f"{row['name']} Fire Station, {row['postcode']}, London, UK"
    try:
        loc = geocode(query)
        if loc:
            print(f"Geocoded: {query} -> ({loc.latitude}, {loc.longitude})")
            return pd.Series({"latitude": loc.latitude, "longitude": loc.longitude})
    except Exception as exc:
        print(f"Geocoding failed for {query}: {exc}")
    return pd.Series({"latitude": None, "longitude": None})
geo_results = stations_df.apply(geocode_station, axis=1)
stations_df = pd.concat([stations_df, geo_results], axis=1)
stations_df

Geocoded: Barking Fire Station, IG11 0BB, London, UK -> (51.5300005, 0.0888737)
Geocoded: Dagenham Fire Station, RM10 7ES, London, UK -> (51.5594685, 0.1570282)
Geocoded: Dowgate Fire Station, EC4R 3UE, London, UK -> (51.510035, -0.0901008)
Geocoded: Shoreditch Fire Station, EC1V 9EY, London, UK -> (51.5266217, -0.0854858)
Geocoded: Stoke Newington Fire Station, N16 0AR, London, UK -> (51.5625722, -0.076866)
Geocoded: Homerton Fire Station, E9 6DL, London, UK -> (51.5486517, -0.0439338)
Geocoded: Harold Hill Fire Station, RM3 8UN, London, UK -> (51.5983747, 0.2235414)
Geocoded: Hornchurch Fire Station, RM11 1SH, London, UK -> (51.5645114, 0.2205803)
Geocoded: Romford Fire Station, RM1 4PL, London, UK -> (51.5936769, 0.1811242)
Geocoded: Wennington Fire Station, RM13 9EE, London, UK -> (51.5062124, 0.2204266)
Geocoded: Holloway Fire Station, N7 7QZ, London, UK -> (51.5615714, -0.1159427)
Geocoded: Islington Fire Station, N1 2TZ, London, UK -> (51.5402671, -0.1023069)
Geocoded: East Ham 

,name,area,postcode,latitude,longitude
0,Barking,Barking and Dagenham,IG11 0BB,51.530000,0.088874
1,Dagenham,Barking and Dagenham,RM10 7ES,51.559469,0.157028
2,Dowgate,City,EC4R 3UE,51.510035,-0.090101
3,Shoreditch,Hackney,EC1V 9EY,51.526622,-0.085486
4,Stoke Newington,Hackney,N16 0AR,51.562572,-0.076866
...,...,...,...,...,...
98,Wandsworth,Wandsworth,SW18 1RL,51.456383,-0.201401
99,Battersea,Wandsworth,SW11 2TL,51.467115,-0.169294
100,Tooting,Wandsworth,SW17 7SQ,51.437827,-0.162705
101,Paddington,Westminster,W2 6NL,51.520072,-0.183319


In [7]:
stations_df

,name,area,postcode,latitude,longitude
0,Barking,Barking and Dagenham,IG11 0BB,51.530000,0.088874
1,Dagenham,Barking and Dagenham,RM10 7ES,51.559469,0.157028
2,Dowgate,City,EC4R 3UE,51.510035,-0.090101
3,Shoreditch,Hackney,EC1V 9EY,51.526622,-0.085486
4,Stoke Newington,Hackney,N16 0AR,51.562572,-0.076866
...,...,...,...,...,...
98,Wandsworth,Wandsworth,SW18 1RL,51.456383,-0.201401
99,Battersea,Wandsworth,SW11 2TL,51.467115,-0.169294
100,Tooting,Wandsworth,SW17 7SQ,51.437827,-0.162705
101,Paddington,Westminster,W2 6NL,51.520072,-0.183319


In [8]:
# save the results to CSV
stations_df.to_csv("Data/output.csv", index=False)